# 示例: 6. (简单)河网水动力模拟示例

**脚本:** `examples/run_network_example.py`

## 目的
演示如何将多个 `MuskingumCungeRouting` 模块组合在一个河网中，进行完整的网络水动力模拟。
- **功能:**
    1.  构建一个包含两条支流和一条干流的“Y”形河网。
    2.  为两个上游源头节点提供不同的入流过程线。
    3.  为干流河段提供侧向入流，以模拟平原产流。
    4.  运行完整的网络模拟。
- **验证:** 最终的图表会展示两个支流的入流过程线，以及在最终总出口处经过演算和叠加后的总出流过程线，验证了`Catchment`类的网络处理能力。

## 1. 环境设置

首先，我们导入所需的库，并把项目的根目录添加到Python的搜索路径中。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
from hydro_model.catchment import Catchment
from hydro_model.routing import MuskingumCungeRouting

## 2. 构建Y形河网

我们使用`Catchment`类来构建一个包含两条支流（`R1`, `R2`）和一条干流（`R3`）的Y形河网。两条支流在节点`N3`处汇合。

In [ ]:
network = Catchment()

for i in range(1, 5):
    network.add_node(f"N{i}")

reach1_params = {'length': 5000, 'slope': 0.0003, 'manning_n': 0.035, 'width': 30.0}
reach2_params = {'length': 4000, 'slope': 0.0004, 'manning_n': 0.04, 'width': 25.0}
reach3_params = {'length': 8000, 'slope': 0.0002, 'manning_n': 0.03, 'width': 50.0}

network.add_reach("R1", "N1", "N3", MuskingumCungeRouting(**reach1_params))
network.add_reach("R2", "N2", "N3", MuskingumCungeRouting(**reach2_params))
network.add_reach("R3", "N3", "N4", MuskingumCungeRouting(**reach3_params))

print("Y形河网构建完成。")

## 3. 定义入流

我们为两个上游源头节点（`N1`和`N2`）分别定义不同的入流过程线，并为干流河段`R3`定义了侧向入流，用以模拟区间汇流。

In [ ]:
T = 40
inflow1 = np.array([0,0,5,15,25,20,15,10,5,3,1,0,0,0,0,0,0,0,0,0] + [0]*20)
inflow2 = np.array([0,0,0,0,0,3,10,20,18,15,10,8,5,3,1,0,0,0,0,0] + [0]*20)
lateral_inflow3 = np.array([1,1,2,3,3,3,2,2,1,1,0,0,0,0,0,0,0,0,0,0] + [0]*20)

headwater_inflows = {"N1": inflow1, "N2": inflow2}
lateral_inflows = {"R3": lateral_inflow3}

print("入流定义完成。")

## 4. 运行模拟

由于这个河网是树状结构（无环），我们可以使用效率更高的`run_simulation`方法，它会先对河网进行拓扑排序，然后按从上游到下游的顺序进行演算。

In [ ]:
print("运行河网模拟...")
reach_q, node_q = network.run_simulation(headwater_inflows, lateral_inflows, T)
print("模拟完成。")

## 5. 可视化结果

我们将两条支流的入流过程线和最终出口（`N4`）的出流过程线绘制在一起。
可以看到，出口的流量过程线是两条支流入流经过各自河段的演算（削峰、坦化），然后在汇合点叠加，再经过干流演算后的最终结果。

In [ ]:
plt.figure(figsize=(15, 7))
timesteps = np.arange(T)

plt.plot(timesteps, headwater_inflows['N1'], 'b--', label='Inflow at N1 (Tributary 1)')
plt.plot(timesteps, headwater_inflows['N2'], 'g--', label='Inflow at N2 (Tributary 2)')
plt.plot(timesteps, node_q['N4'], 'r-', linewidth=2, label='Outflow at N4 (Final Outlet)')

plt.title('River Network Hydrodynamic Simulation')
plt.xlabel('Time Step (days)')
plt.ylabel('Flow (m³/s)')
plt.legend()
plt.grid(True)

output_dir = '../examples/results/'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'network_simulation_plot.png')
plt.savefig(output_path)
print(f"\n河网模拟图已保存到 {output_path}")
plt.show()